In [4]:
2+2

4

In [5]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re


In [6]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [ ]:
result = pd.read_parquet("vjepa_ssv2_facebook_vjepa2-vitg-fpc64-384-ssv2_meta.parquet")
print(result.shape)
del result['top5_classes']
result.head(3)

(2543, 14)


,id,video_id,text_label,template,placeholders,split,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,69703,69703,moving pen up,Moving [something] up,['pen'],val,1,1,Moving [something] up,0.92,"[1, 4, 6, 5, 0]",True,True
1,215371,215371,tearing a piece of paper into two pieces,Tearing [something] into two pieces,['a piece of paper'],val,9,9,Tearing [something] into two pieces,1.00,"[9, 0, 1, 5, 3]",True,True
2,29495,29495,moving bowl up,Moving [something] up,['bowl'],val,1,1,Moving [something] up,1.00,"[1, 8, 6, 2, 9]",True,True


# main result

In [9]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

{'total_evaluated': 2543,
 'top1_accuracy': np.float64(0.91978),
 'top1_balanced_accuracy': np.float64(0.910553),
 'top5_accuracy': np.float64(0.997641),
 'top5_balanced_accuracy': np.float64(0.997568)}

In [11]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.80,1.00
Moving [something] up,0.96,1.00
Piling [something] up,0.93,1.00
Plugging [something] into [something],0.92,1.00
Pushing [something] with [something],0.96,0.99
Putting [something] next to [something],0.94,1.00
Putting [something] on a surface,0.77,0.99
Putting [something] onto [something],0.84,1.00
Showing a photo of [something] to the camera,0.98,1.00


In [2]:
from transformers import AutoConfig

# Loads instantly from local cache (~/.cache/huggingface/hub/...)
config = AutoConfig.from_pretrained("facebook/vjepa2-vitg-fpc64-384-ssv2")

id2label = config.id2label  # Dict: {0: 'Approaching something...', 1: '...', ...}
label2id = config.label2id  # Dict: {'Approaching something...': 0, ...}

# Example: Print class index 0
print(id2label[0])

Approaching [something] with your camera


In [3]:
print(id2label[173])

Wiping [something] off of [something]
